# T4 — Targeted Re-Run of Stage S3 (Correctness & KAT) — Post-Packing-Fix

This notebook re-validates **only** the H4 KAT and random-tensor correctness checks that
failed in run `2026-07-30_165546` due to a harness input-packing bug (`& 0x7FFFFFFF`
dropping bit 31). Compile audit, fused GEMM, telemetry, and ncu already PASSED and are
not re-run here.

**Runtime → Change runtime type → T4 GPU**, then **Run all**. Upload
`research/harness/empirical/run_empirical.py` when prompted.


In [ ]:
#@title 1) Confirm Tesla T4
!nvidia-smi
import torch
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('SM cap :', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'NONE')


In [ ]:
#@title 2) Config
import os
REPO_URL = 'https://github.com/kridaydave/t4-cuda'  #@param {type:'string'}
BRANCH   = 'main'                                     #@param {type:'string'}
WORKDIR  = '/content/repo'                            #@param {type:'string'}
USE_TOKEN = False                                     #@param {type:'boolean'}
TOKEN = ''                                            #@param {type:'string'}
REPO_URL_AUTH = REPO_URL.replace('https://', f'https://{TOKEN}@') if (USE_TOKEN and TOKEN) else REPO_URL
print(REPO_URL, BRANCH, WORKDIR)


In [ ]:
#@title 3) Clone repo
import os, shutil, subprocess
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
rc = subprocess.call(['git','clone','--depth','1','-b',BRANCH,REPO_URL_AUTH,WORKDIR])
assert rc == 0, 'clone failed'
print('HEAD:', subprocess.check_output(['git','-C',WORKDIR,'rev-parse','HEAD']).decode().strip())


### 4) Upload the FIXED run_empirical.py
The clone contains the *old* (bugged) harness. We must overwrite it with the fixed
version from your local `research/harness/empirical/run_empirical.py`.


In [ ]:
#@title 4) Overwrite harness with fixed run_empirical.py
import os
from google.colab import files  # type: ignore
print('Select local file: research/harness/empirical/run_empirical.py')
up = files.upload()
assert 'run_empirical.py' in [os.path.basename(k) for k in up.keys()], 'must upload run_empirical.py'
for name, data in up.items():
    if os.path.basename(name) == 'run_empirical.py':
        tgt = os.path.join(WORKDIR,'research','harness','empirical','run_empirical.py')
        open(tgt,'wb').write(data)
        print('installed fixed harness ->', tgt, f'({len(data)} bytes)')


In [ ]:
#@title 5) Run ONLY Stage S3 (skip ncu, skip telemetry, skip build)
import os, subprocess
RESEARCH = os.path.join(WORKDIR, 'research')
env = dict(os.environ); env['PYTHONPATH'] = os.path.join(RESEARCH,'src') + ':' + RESEARCH
p = subprocess.run(['python3','-c',
    'import sys, os\n'
    f'sys.path.insert(0, {os.path.join(RESEARCH,'harness','empirical')!r})\n'
    'os.chdir({!r})\n'.format(RESEARCH)
    'import run_empirical as R\n'
    'import pathlib, datetime as dt\n'
    'ts = dt.datetime.now().strftime("%Y-%m-%d_%H%M%S")\n'
    'rd = R.RESULTS_ROOT / ts; rd.mkdir(parents=True, exist_ok=True)\n'
    'log = open(rd / "run.log", "w")\n'
    'sys.stdout = R.Tee(sys.__stdout__, log); sys.stderr = R.Tee(sys.__stderr__, log)\n'
    'expected = R._load_expected()\n'
    'ctx = {"run_dir": rd, "expected": expected, "artifacts": {}}\n'
    'R._banner("TARGETED RE-RUN: Stage S3 only")\n'
    'r = R.stage_gpu_correctness(ctx)\n'
    'R._banner(f"S3 STATUS: {r.status}")\n'
    'log.close(); sys.stdout = sys.__stdout__; sys.stderr = sys.__stderr__\n'
    'print("saved:", rd)'
    ],
    cwd=RESEARCH, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)
print('RETURN CODE:', p.returncode)


In [ ]:
#@title 6) Show S3 verdict + download mini-bundle
import os
RESROOT = os.path.join(WORKDIR, 'research', 'results')
runs = sorted([d for d in os.listdir(RESROOT) if os.path.isdir(os.path.join(RESROOT,d))])
LATEST = runs[-1]
print('new run dir:', LATEST)
print(open(os.path.join(RESROOT, LATEST, 'run.log')).read())
import subprocess
subprocess.run(['tar','czf', os.path.join(RESROOT, LATEST + '.tar.gz'), '-C', RESROOT, LATEST])
from google.colab import files  # type: ignore
files.download(os.path.join(RESROOT, LATEST + '.tar.gz'))


---
### Interpretation
- If all four S3 sub-tests print **CONFIRM**, the only blocker in the
  original FAIL verdict was the packing bug. Paste the output back and I'll merge
  `EMPIRICALLY_VERIFIED_ON_HARDWARE` for H4, fused W4A16, H7-math, and H9-math.
- If random-allclose still shows ~0.03 diff after the fix, that is FP16-mantissa
  rounding in the scale/bias `fma.rn.f16x2` and we quantify it as such.
